# Towards Convex Formulations in Calibration

This Jupyter notebook details convex formulations for problems in camera calibration and geometric perception.  We study these methods by using the representative problem of __Similarity Registration under Known Correspondences__ (SRKC), which serves as a realistic component of an iterative closest point (ICP) solver for similarity point registration or LiDAR calibration.

I developed this notebook in tandem with a technical report available on GitHub at [josephrhobbs/towards-convex-calibration](https://github.com/josephrhobbs/towards-convex-calibration).

__License__: MIT

## Imports

In [ ]:
!pip install open3d

In [ ]:
from abc import ABC, abstractmethod
import cvxpy as cp
from matplotlib import pyplot as plt
import numpy as np
import open3d as o3d
import scipy as sp
from time import perf_counter
from tqdm import tqdm
import urllib.request

## Utilities

### Construct Skew-Symmetric Matrix

In [ ]:
def hat(axis_angle: np.ndarray) -> np.ndarray:
    """
    Construct a skew-symmetric matrix from an axis-angle vector.
    """
    return np.array([
        [0, -axis_angle[2], axis_angle[1]],
        [axis_angle[2], 0, -axis_angle[0]],
        [-axis_angle[1], axis_angle[0], 0],
    ])

### Convert Axis-Angle Vectors to Rotation Matrices

In [ ]:
def axis_angle_to_rotation(axis_angle: np.ndarray) -> np.ndarray:
    """
    Convert an axis-angle vector into a rotation matrix.
    """
    # Construct skew-symmetric matrix
    skewsym = hat(axis_angle)

    # Exponentiate
    R = sp.linalg.expm(skewsym)

    return R.real

### Compute Triple Geodesic Distance across Sim(3)

In [ ]:
def to_angle(R: np.ndarray):
    """
    Compute the geodesic distance of a rotation matrix
        from the identity matrix.
    """
    w = sp.linalg.logm(R).real
    a = np.array([-w[1, 2], w[0, 2], -w[0, 1]])
    return np.linalg.norm(a)

def triple_geodesic(
    scale1: float,
    rotation1: np.ndarray,
    translation1: np.ndarray,
    scale2: float,
    rotation2: np.ndarray,
    translation2: np.ndarray,
    epsilon: float = 1e-9,
):
    """
    Compute a "triple geodesic distance" across Sim(3).
    """
    scale1, scale2 = abs(scale1), abs(scale2)
    d1 = np.power(np.log(scale2 / (scale1 + epsilon)), 2)
    d2 = np.power(to_angle(rotation2.T @ rotation1), 2)
    d3 = np.power(np.linalg.norm(translation2 - translation1), 2)
    return np.sqrt(d1 + d2 + d3)

### Convert Quaternions to Rotation Matrices

In [ ]:
def L_matrix(q: np.ndarray) -> np.ndarray:
    """
    Compute the linear operator corresponding to quaternion
        left-multiplication.
    """
    s, v = q[0], q[1:]
    return np.block([
        [np.array([[s]]), -v.reshape((1, 3))],
        [v.reshape((3, 1)), s*np.eye(3) + hat(v)],
    ])

def R_matrix(q: np.ndarray) -> np.ndarray:
    """
    Compute the linear operator corresponding to quaternion
        right-multiplication.
    """
    s, v = q[0], q[1:]
    return np.block([
        [np.array([[s]]), -v.reshape((1, 3))],
        [v.reshape((3, 1)), s*np.eye(3) - hat(v)],
    ])

In [ ]:
def quaternion_to_rotation(q: np.ndarray) -> np.ndarray:
    """
    Construct a rotation matrix from a quaternion.
    """
    H = np.block([
        [np.zeros((1, 3))],
        [np.eye(3)],
    ])
    return H.T @ L_matrix(q) @ R_matrix(q).T @ H

### Plot 3D Data

In [ ]:
def plot_3d_data(
    point_clouds: list[np.ndarray],
    labels=None,
    title="Point Clouds",
):
    """
    Given a collection of point clouds, plot them.
    """
    # Construct plot
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # Process labels
    if labels is None:
        labels = [None] * len(point_clouds)

    # Plot each point cloud
    for pc, label in zip(point_clouds, labels):
        ax.scatter(pc[0, :], pc[1, :], pc[2, :], label=label)

    if any(l is not None for l in labels):
        plt.legend()
    plt.suptitle(title)
    plt.show()

## Similarity Registration under Known Correspondences (SRKC)

### Define Solver Base Class

In [ ]:
class Solver(ABC):
    """
    Abstract base class for SRKC solver.
    """
    def __init__(self):
        """
        Initialize this solver.
        """
        pass

    @abstractmethod
    def solve(
        self,
        x: np.ndarray,
        y: np.ndarray,
    ) -> tuple[float, np.ndarray, np.ndarray, float]:
        pass

### Define SRKC Solver

In [ ]:
class VanillaSRKCSolver(Solver):
    """
    Solver for Similarity Registration under Known Correspondences (SRKC).

    This solver, given two point clouds, estimates the similarity transform
        that relates them.
    """
    def solve(
        self,
        x: np.ndarray,
        y: np.ndarray,
    ) -> tuple[float, np.ndarray, np.ndarray, float]:
        """
        Given two point clouds `x` and `y`, solve for the similarity transform
            that relates them.
        """
        start = perf_counter()

        # Compute point cloud centroids
        xmean = np.mean(x, axis=1)
        ymean = np.mean(y, axis=1)

        # Compute centered point clouds
        a = x - xmean.reshape(3, 1)
        b = y - ymean.reshape(3, 1)

        # Construct matrix `H`
        H = np.sum(np.einsum("ib,jb->bij", a, b), axis=0)

        # Construct decision variable (rotation)
        R = cp.Variable((3, 3))

        # Construct convex optimization
        objective = cp.trace(R @ H)
        constraint = cp.bmat([
            [np.eye(3), R],
            [R.T, np.eye(3)]
        ]) >> 0
        program = cp.Problem(
            cp.Maximize(objective),
            [constraint],
        )

        # Solve optimization and extract optimal rotation
        program.solve()
        assert program.status == "optimal"
        R = R.value

        # Project onto SO(3)
        U, _, Vh = np.linalg.svd(R)
        R = U @ np.diag([1, 1, np.linalg.det(U @ Vh)]) @ Vh

        # Construct scale and translation
        numer = np.sum(np.einsum("ib,ij,jb->b", b, R, a))
        denom = np.sum(np.einsum("ib,ib->b", a, a))
        s = numer/denom
        t = np.mean(y - s * (R @ x), axis=1)

        duration = perf_counter() - start

        return s, R, t, duration

### Define Constrained SRKC Solver

In [ ]:
class ConstrainedSRKCSolver(Solver):
    """
    Solver for Constrained Similarity Registration
        under Known Correspondences (C-SRKC).

    This solver, given two point clouds, estimates the similarity transform
        that relates them.
    """
    def __init__(
        self,
        s_min=None,
        s_max=None,
    ):
        """
        Initialize this constrained SRKC solver.
        """
        self.s_min = s_min
        self.s_max = s_max

    def solve(
        self,
        x: np.ndarray,
        y: np.ndarray,
    ) -> tuple[float, np.ndarray, np.ndarray, float]:
        """
        Given two point clouds `x` and `y`, solve for the similarity transform
            that relates them.
        """
        start = perf_counter()

        # Compute point cloud centroids
        xmean = np.mean(x, axis=1)
        ymean = np.mean(y, axis=1)

        # Compute centered point clouds
        a = x - xmean.reshape(3, 1)
        b = y - ymean.reshape(3, 1)

        # Vector of ones
        e = np.ones((3, 1))

        # Construct decision variables
        Q = cp.Variable((3, 3))
        g = cp.Variable()
        t = cp.mean(
            y - Q @ x,
            axis=1,
        )

        # Objective
        objective = 0
        Qa = Q @ a
        objective = g * cp.trace(a @ a.T) - 2 * cp.trace(b @ Qa.T)

        # Constraints
        g_pos = g >= 0
        lmi = cp.bmat([
            [g * np.eye(3), Q],
            [Q.T, np.eye(3)]
        ]) >> 0
        constraints = [g_pos, lmi]

        # Box constraints on scale
        if self.s_min is not None:
            constraints.append(g >= self.s_min**2)
        if self.s_max is not None:
            constraints.append(g <= self.s_max**2)

        # Construct convex optimization
        program = cp.Problem(
            cp.Minimize(objective),
            constraints,
        )

        # Solve optimization
        program.solve()
        assert program.status == "optimal"

        # Project onto SO(3)
        Q = Q.value
        U, _, Vh = np.linalg.svd(Q)
        R = U @ np.diag([1, 1, np.linalg.det(U @ Vh)]) @ Vh

        # Get size and translation
        s = np.sqrt(g.value)
        t = np.mean(y - s * (R @ x), axis=1)

        duration = perf_counter() - start

        return s, R, t, duration

### Define Quaternion SKRC Solver

In [ ]:
class QuaternionSRKCSolver(Solver):
    """
    Solver for Quaternion Similarity Registration
        under Known Correspondences (q-SRKC).

    This solver, given two point clouds, estimates the similarity transform
        that relates them using Davenport's q-method.
    """
    def __init__(self, epsilon=1e-9, iterations=250):
        """
        Initialize this q-SRKC solver.
        """
        self.eps = epsilon
        self.iter = iterations

    def solve(
        self,
        x: np.ndarray,
        y: np.ndarray,
    ) -> tuple[float, np.ndarray, np.ndarray, float]:
        """
        Given two point clouds `x` and `y`, solve for the similarity transform
            that relates them.
        """
        start = perf_counter()

        # Compute point cloud centroids
        xmean = np.mean(x, axis=1)
        ymean = np.mean(y, axis=1)

        # Compute centered point clouds
        a = x - xmean.reshape(3, 1)
        b = y - ymean.reshape(3, 1)

        # Construct matrix `H`
        H = np.sum(np.einsum("ib,jb->bij", a, b), axis=0)

        # Construct Davenport's K matrix
        sigma = np.trace(H)
        z = np.array([
            H[1, 2] - H[2, 1],
            H[2, 0] - H[0, 2],
            H[0, 1] - H[1, 0],
        ]).reshape((3, 1))
        S = H + H.T
        K = np.block([
            [sigma * np.ones((1, 1)), z.T],
            [z, S - sigma*np.eye(3)],
        ])

        # Solve using power iteration
        q = np.array([1, 0, 0, 0])
        for _ in range(self.iter):
            next_q = K @ q
            next_q = next_q / np.linalg.norm(next_q)
            if np.linalg.norm(next_q - q) < self.eps:
                break
            q = next_q

        # Get rotation
        R = quaternion_to_rotation(q)

        # Construct scale and translation
        numer = np.sum(np.einsum("ib,ij,jb->b", b, R, a))
        denom = np.sum(np.einsum("ib,ib->b", a, a))
        s = numer/denom
        t = np.mean(y - s * (R @ x), axis=1)

        duration = perf_counter() - start

        return s, R, t, duration

### Test Solver

In [ ]:
# Generate a random point cloud
pc = np.random.normal(size=(3, 1000)) / np.sqrt(3)

# Iterations per noise level
iterations = 40

# Calculate mean error over various noise levels
noise_db = np.linspace(-80, 20, 100)
errors = []

# Calculate mean runtime duration
durations = []

for n in tqdm(noise_db):
    # Calculate noise level
    signal = np.linalg.norm(np.std(pc, axis=1))
    noise_level = np.power(10, n/20) * signal
    e = []
    for _ in range(iterations):
        # Random rotation
        q = np.random.normal(size=(4,))
        q /= np.linalg.norm(q)
        R = quaternion_to_rotation(q)

        # Random scale
        s = np.exp(np.random.normal())

        # Random translation
        t = np.random.normal(size=(3,))

        # Measurement noise
        noise = noise_level * np.random.normal(size=pc.shape)

        # Transformed point cloud
        tc = s * (R @ pc) + t.reshape((3, 1)) + noise

        # Solve for transformation
        # No SRKC constraints
        solver = QuaternionSRKCSolver()
        s_est, R_est, t_est, duration = solver.solve(pc, tc)
        durations.append(duration)

        # Calculate triple geodesic error
        error = triple_geodesic(
            s, R, t,
            s_est, R_est, t_est,
        )
        e.append(error)
    errors.append(e)

# Calculate mean error
mean_error = np.mean(np.array(errors), axis=1)

# Calculate p95 latency
p95_latency = 1e3*np.percentile(np.array(durations), 95)

# Plot mean error over noise level
fig = plt.figure()
ax = fig.add_subplot()
ax.semilogy(noise_db, mean_error)
plt.xlabel("Noise [dB]")
plt.ylabel("Triple Geodesic Error")
plt.suptitle(
    f"Measurement Noise vs. Solver Error "
    f"(p95 latency: {p95_latency:.3f} ms)"
)
plt.grid(True)
plt.show()

### Create Bunny Visualization for Repository

In [ ]:
BUNNY_URL = "https://gist.githubusercontent.com/bigsnarfdude/" \
            "ac6b9911f34630d5b24508e628cfd0b1/raw/" \
            "8b1ae1fa66b576bc780e7cbc3550da51ffdd80af/bunny.pcd"

# Load Stanford bunny
urllib.request.urlretrieve(BUNNY_URL, "bunny.pcd")
pcd = o3d.io.read_point_cloud("bunny.pcd")
pc = axis_angle_to_rotation([np.pi/2, 0, 0]) @ np.asarray(pcd.points).T
scale = np.linalg.norm(np.std(pc, axis=1))
pc /= scale

# Random rotation
q = np.random.normal(size=(4,))
q /= np.linalg.norm(q)
R = quaternion_to_rotation(q)

# Random scale
s = np.exp(0.5 * np.random.normal())

# Random translation
t = 0.5 * np.random.normal(size=(3,)) + np.array([0, 0, -1])

# Measurement noise
noise = 1e-3 * np.random.normal(size=pc.shape)

# Transformed point cloud
tc = s * (R @ pc) + t.reshape((3, 1)) + noise

# Solve
solver = ConstrainedSRKCSolver(
    s_min=0.75,
    s_max=1.33,
)
s_est, R_est, t_est, _latency = solver.solve(pc, tc)

# Shift estimate over for visibility
t_est += np.array([2, 0, 0])

# Approximate target point cloud
ec = s_est * (R_est @ pc) + t_est.reshape((3, 1))

plot_3d_data(
    [pc, tc, ec],
    labels=["Source", "Target", "Estimate (Translated)"],
    title="Similarity Registration under Known Correspondences",
)